# 🃏 The Mind — LLMs + RL en Kaggle

| GPU | VRAM | Modelo recomendado | 4-bit |
|-----|------|--------------------|-------|
| T4  | 16GB | Qwen2.5-7B-Instruct | No |
| P100 | 16GB | Qwen2.5-7B-Instruct | No |
| T4 x2 | 32GB | Qwen2.5-14B-Instruct | No |

> **TPU**: no compatible directamente con este código (necesita torch_xla). Usa GPU.

In [ ]:
# ── 1. Clonar repo e instalar dependencias ────────────────────────────────────
import os, sys

!git clone https://github.com/JesusCristoII/The-Mind-LLMs
%cd The-Mind-LLMs
sys.path.append(os.getcwd())

# Instalar solo lo que no está en Kaggle
!pip install -q peft>=0.10.0 bitsandbytes>=0.43.0

In [ ]:
# ── 2. Detección de hardware ──────────────────────────────────────────────────
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No se detectó GPU. En Kaggle: Settings > Accelerator > GPU T4 x2")

DEVICE = "cuda"
n_gpus   = torch.cuda.device_count()
vram_gb  = sum(torch.cuda.get_device_properties(i).total_memory for i in range(n_gpus)) / 1e9
gpu_name = torch.cuda.get_device_name(0)
compute  = torch.cuda.get_device_capability(0)

print(f"GPU: {gpu_name} x{n_gpus}")
print(f"VRAM total: {vram_gb:.1f} GB")
print(f"Compute capability: sm_{compute[0]}{compute[1]}")

# Selección automática de modelo según VRAM disponible
if vram_gb >= 28:
    MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"
    USE_4BIT   = False
elif vram_gb >= 14:
    MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
    USE_4BIT   = False
elif vram_gb >= 8:
    MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
    USE_4BIT   = True
else:
    MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
    USE_4BIT   = False

# Flash Attention solo en Ampere+ (sm_80+), no en T4 (sm_75)
USE_FLASH_ATTN = compute[0] >= 8

print(f"\nModelo seleccionado: {MODEL_NAME}")
print(f"4-bit: {USE_4BIT} | Flash Attention: {USE_FLASH_ATTN}")

In [ ]:
# ── 3. Cargar modelo ──────────────────────────────────────────────────────────
import logging
from agents import load_base_model
from utils import setup_logging

setup_logging(level="WARNING")  # menos ruido en Kaggle
logging.getLogger("agents").setLevel(logging.WARNING)

base_model, tokenizer = load_base_model(
    model_name=MODEL_NAME,
    device="auto",
    use_4bit=USE_4BIT,
    use_flash_attention=USE_FLASH_ATTN,
)
print(f"Modelo cargado: {base_model.num_parameters()/1e9:.2f}B parámetros")

In [ ]:
# ── 4. Asegurar chat_template (necesario con transformers 5.x) ────────────────
QWEN_CHAT_TEMPLATE = (
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}<|im_start|>system\n{{ message['content'] }}<|im_end|>\n"
    "{% elif message['role'] == 'user' %}<|im_start|>user\n{{ message['content'] }}<|im_end|>\n"
    "{% elif message['role'] == 'assistant' %}<|im_start|>assistant\n{{ message['content'] }}<|im_end|>\n"
    "{% endif %}{% endfor %}"
    "{% if add_generation_prompt %}<|im_start|>assistant\n{% endif %}"
)

if getattr(tokenizer, "chat_template", None) is None:
    tokenizer.chat_template = QWEN_CHAT_TEMPLATE
    print("Chat template asignado manualmente ✓")
else:
    print("Chat template ya presente ✓")

# Test
test = tokenizer.apply_chat_template(
    [{"role": "user", "content": "hola"}],
    tokenize=False, add_generation_prompt=True
)
print(f"Test: {test[:60]}...")

In [ ]:
# ── 5. Crear agentes ──────────────────────────────────────────────────────────
from agents import create_agents

NUM_PLAYERS = 4
SHARED_LORA = True

# Con GPU de 16GB podemos permitirnos r=16
LORA_R = 16 if vram_gb >= 14 else 8

agents = create_agents(
    model=base_model,
    tokenizer=tokenizer,
    num_players=NUM_PLAYERS,
    device=DEVICE,
    lora_r=LORA_R,
    shared_lora=SHARED_LORA,
)
print(f"{NUM_PLAYERS} agentes creados | LoRA r={LORA_R} | Compartida: {SHARED_LORA}")
agents[0].model.print_trainable_parameters()

In [ ]:
# ── 6. SFT previo al RL ───────────────────────────────────────────────────────
from sft_trainer import run_sft, verify_sft_quality

sft_results = run_sft(
    agents=agents,
    tokenizer=tokenizer,
    dataset_path="sft_dataset.json",
    epochs=3,
    batch_size=4,      # más batch con 16GB
    lr=2e-4,
    max_length=512,
    save_dir="/kaggle/working/checkpoints/sft",
    device=DEVICE,
    shared_lora=SHARED_LORA,
)
print(f"SFT completado. Loss final: {sft_results['final_loss']:.4f}")

In [ ]:
# ── 7. Verificar SFT ─────────────────────────────────────────────────────────
json_ok_rate = verify_sft_quality(agents, tokenizer, num_samples=5, device=DEVICE)

if json_ok_rate >= 0.8:
    print(f"\n✓ SFT exitoso ({json_ok_rate:.0%} JSON válidos). Listo para RL.")
else:
    print(f"\n⚠ Solo {json_ok_rate:.0%} JSON válidos. Considera más epochs de SFT.")

In [ ]:
# ── 8. Configuración RL ───────────────────────────────────────────────────────
from environment import TheMindEnv
from trainer import GRPOTrainer, TrainerConfig
from utils import TrainingMetrics, LanguageAnalyzer

env = TheMindEnv(num_players=NUM_PLAYERS)

config = TrainerConfig(
    num_episodes=500,
    num_levels=3,
    group_size=4,
    lr=5e-6,           # LR más bajo para modelos grandes
    kl_coeff=0.01,
    entropy_bonus=0.01,
    warmup_episodes=20,
    max_turns_per_episode=150,
    messages_per_turn=True,
    device=DEVICE,
    accumulate_grad_steps=4,
    checkpoint_every=50,
)

metrics      = TrainingMetrics()
lang_analyzer = LanguageAnalyzer()

trainer = GRPOTrainer(agents=agents, env=env, config=config)
print("Trainer listo ✓")

In [ ]:
# ── 9. Entrenamiento RL ───────────────────────────────────────────────────────
# Kaggle corta la sesión después de ~9h en GPU.
# Los checkpoints se guardan en /kaggle/working/checkpoints/ cada 50 episodios.
# Para descargarlos: File > Output > Download.

import os
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)

print("Iniciando entrenamiento RL...")
print(f"GPU: {gpu_name} | Modelo: {MODEL_NAME}")
print("Los checkpoints se guardan en /kaggle/working/checkpoints/")
print("="*60)

try:
    trainer.train(
        metrics=metrics,
        lang_analyzer=lang_analyzer,
        verbose=True,
    )
except KeyboardInterrupt:
    print("\nEntrenamiento interrumpido.")

In [ ]:
# ── 10. Guardar resultados finales ────────────────────────────────────────────
from utils import save_checkpoint

save_checkpoint(
    agents=agents,
    metrics=metrics,
    episode=trainer.episode_count,
    output_dir="/kaggle/working/checkpoints",
)

metrics.plot("/kaggle/working/training_metrics.png")
lang_analyzer.save_log("/kaggle/working/language_log.json")

print("\nArchivos guardados en /kaggle/working/:")
print("  checkpoints/  — pesos LoRA por episodio")
print("  training_metrics.png — gráficas")
print("  language_log.json    — mensajes de los agentes")
print("\nDescárgalos desde: Kaggle > Output > Download")

In [ ]:
# ── 11. Análisis del lenguaje emergente ───────────────────────────────────────
metrics.print_summary()
lang_analyzer.print_report()

print("\nÚltimos 15 mensajes de los agentes:")
for msg in lang_analyzer.message_log[-15:]:
    print(f"  Ep {msg['episode']:3d} | J{msg['player']}: {msg['message']}")

In [ ]:
# ── 12. [OPCIONAL] Reanudar desde checkpoint ──────────────────────────────────
# Ejecuta esta celda si quieres continuar un entrenamiento previo.
# Sube los checkpoints a Kaggle como Dataset y monta la ruta.

# from utils import load_checkpoint, list_checkpoints
# 
# CHECKPOINT_DIR = "/kaggle/input/the-mind-checkpoints"  # Dataset subido a Kaggle
# list_checkpoints(CHECKPOINT_DIR)
# 
# RESUME_FROM = 200  # episodio desde el que reanudar
# optimizers = [torch.optim.AdamW(a.model.parameters(), lr=config.lr) for a in agents]
# metrics = load_checkpoint(agents, episode=RESUME_FROM,
#                           output_dir=CHECKPOINT_DIR, optimizers=optimizers)
# trainer = GRPOTrainer(agents=agents, env=env, config=config, optimizers=optimizers)
# trainer.episode_count = RESUME_FROM
# trainer.train(metrics=metrics, lang_analyzer=lang_analyzer)